# Load libraries and enviornment variables

In [78]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai  import ChatOpenAI
from langchain.prompts import PromptTemplate, FewShotChatMessagePromptTemplate
from openai import OpenAI, api_key
from dotenv import load_dotenv
import os
import json
from typing import List, Literal
from typing_extensions import Annotated
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain.tools.render import format_tool_to_openai_function
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.runnables import RunnableLambda, RunnableMap, RunnablePassthrough, RunnableParallel, RunnableSequence
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser 
from langchain.output_parsers import JsonOutputToolsParser
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser
from pydantic import BaseModel, Field, field_validator, model_validator
import requests
from langchain.schema import AIMessage, SystemMessage, HumanMessage
from langchain.callbacks import get_openai_callback
from concurrent.futures import ThreadPoolExecutor
from langchain.prompts import MessagesPlaceholder
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.output_parsers import PydanticOutputParser
from langchain.agents.format_scratchpad import format_to_openai_functions
from langchain.schema.agent import AgentFinish
from langchain.agents import AgentExecutor
import time
import openpyxl
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import re
import ast

In [2]:
# this module contains all functions and classes used in this notebook
from functions_openai import *

## Load OpenAI API key

In [3]:
# Load the .env file
load_dotenv()

# Retrieve the API key from the environment variables
api_key = os.getenv("OPENAI_API_KEY")

# Test if the key was loaded correctly
print("OpenAI API Key loaded:", api_key is not None)

OpenAI API Key loaded: False


## Load PECOS criteria from a .yaml file

In [4]:
import yaml

# Load YAML file - use template to format pecos criteria
pecos_filename = "pecos_criteria.yaml"
with open(pecos_filename, "r") as file:
    config = yaml.safe_load(file)

# Access variables from YAML
population = config["population"]
exposure = config["exposure"] 
comparison = config["comparison"]
outcome = config["outcome"]
study_type = config["study_type"]     

## Load few-shot examples

In [5]:
# function to load examples from text files with python structures
def load_examples(filename):
    with open(filename, "r", encoding="utf-8") as f:
        text = f.read()

    # Remove BOM if present
    text = text.lstrip("\ufeff")
    return ast.literal_eval(text)

# Load examples from text file, use the provided template for formatting. Use 4 examples
examples_filename = "examples.txt"
examples = load_examples(examples_filename) 

# Define workflow: fewshot_mid – Fewshot-prompt mid (Fs-M)

In [6]:
llm = "gpt-4.1-mini-2025-04-14"

In [7]:
system_fewshot_prompt = f"""You are conducting literature screening for systematic reviews and are helpful and thorough. Your task is to evaluate research studies and determine whether they should be included into the review stage. 

To determine eligibility of a study for inclusion, each of the following five study features has to be scrutinized for the following criteria:

*Study population*: {population}

*Exposure*: {exposure}

Comparator or control condition of the *Comparison*: {comparison}

*Outcome*: {outcome}

*Study Type*: {study_type}

You will read the title and abstract of a study, and then decide for each of its five study features if they meet the eligibility criteria. You will call the ValidateForInclusion tool and fill out the dictionary by depositing a brief rationale for your decision in the key "reason", and your actual decision in the key "decision" using either 'True', 'False', or 'NotEvaluable', if you are not sure whether to include or exclude the study based on this study feature. You will be shown some examples.
"""

In [8]:
parse_results = RunnableLambda(
    lambda x : get_message(x)
)

parse_results_langchain = RunnableLambda(
    lambda x : get_message_langchain(x)
)

In [9]:
fewshot_chain = parse_results | OpenAIFunctionsAgentOutputParser() | format_zeroshot_results

In [10]:
toolwrapper = convert_to_openai_tool(ValidateForInclusion, strict=True)
toolwrapper["name"] = "ValidateForInclusion"
tools = [toolwrapper]

# Load literature

## Option A: Read from Excel file

In [113]:
# Provide the path to the SR records Excel file.
# The file should have a Title and Abstract column.

path_SR_abs = "abs-screen.xlsx"

df = pd.read_excel(path_SR_abs)

In [114]:
df

,Title,Abstract
0,Magnetic field variability and cognitive perfo...,OBJECTIVE: To explore whether ambient magnetic...
1,Microbial adaptation to synthetic gravity in o...,OBJECTIVE: To assess whether microbial communi...
2,Effects of algorithmic soundscapes on plant gr...,OBJECTIVE: To determine whether exposure to al...


In [115]:
# get read of any "new lines" or "tabs"
df.replace({'\n': ' ', '\r': ' ', '\t': ' '}, regex=True, inplace=True)

# Create columns with explicit labels
df['Title_expl']= "Title: " + df['Title']
df['Abstract_expl']= "Abstract: " + df['Abstract']

In [116]:
#concatenate Title and Abstract into a single column, into a new data frame
data = pd.DataFrame()
data['Study_content'] = df['Title_expl'].astype(str).str.cat([df['Abstract_expl'].astype(str)], sep='. ')

In [117]:
data

,Study_content
0,Title: Magnetic field variability and cognitiv...
1,Title: Microbial adaptation to synthetic gravi...
2,Title: Effects of algorithmic soundscapes on p...


In [118]:
# generate a TiAb dictionary
abstracts_dict = [{"input": abstract} for abstract in data.Study_content.to_list()]

## Option B: Read from .ris file

In [89]:
# Provide the path to the SR records ris file.
# The file should have a TI and an AB header.

path_SR_abs = "abs-screen.ris"

df = read_ris(path_SR_abs)

In [90]:
df

,AB,AN,DA,DP,ET,IS,J2,KW,LA,LB,N1,PY,SN,SP,ST,T2,TI,VL,ID
0,OBJECTIVE: To explore whether ambient magnetic...,1000001,Jan,Nlm,2031/01/10,1,Journal of speculative computational science,Magnetic Fields Neural Networks Simulation Cog...,eng,fulltext,SSC-2030-01,2031,0000-0001 (Print) 0000-0001,1-5,Magnetic field variability and cognitive perfo...,J Spec Comput Sci,Magnetic field variability and cognitive perfo...,12,50001
1,OBJECTIVE: To assess whether microbial communi...,1000002,Apr,Nlm,2042/04/03,2,International journal of astro-biology experim...,Microbiology Gravity Simulation Evolution Bior...,eng,fulltext,ABE-2041-07,2042,0000-0002 (Print) 0000-0002,77-82,Microbial adaptation to synthetic gravity in o...,Int J Astro-Biol Exp,Microbial adaptation to synthetic gravity in o...,5,50002
2,OBJECTIVE: To determine whether exposure to al...,1000003,Oct,Nlm,2037/10/21,4,Journal of experimental bioacoustics,Bioacoustics Plant Growth Algorithms Sound Exp...,eng,fulltext,JEB-2037-19,2037,0000-0003 (Print) 0000-0003,210-214,Effects of algorithmic soundscapes on plant gr...,J Exp Bioacoust,Effects of algorithmic soundscapes on plant gr...,9,50003


In [91]:
# get read of any "new lines" or "tabs"
df.replace({'\n': ' ', '\r': ' ', '\t': ' '}, regex=True, inplace=True)

# Create columns with explicit labels
df['Title_expl']= "Title: " + df['TI']
df['Abstract_expl']= "Abstract: " + df['AB']

In [92]:
#concatenate Title and Abstract into a single column, into a new data frame
data = pd.DataFrame()
data['Study_content'] = df['Title_expl'].astype(str).str.cat([df['Abstract_expl'].astype(str)], sep='. ')

In [93]:
data

,Study_content
0,Title: Magnetic field variability and cognitiv...
1,Title: Microbial adaptation to synthetic gravi...
2,Title: Effects of algorithmic soundscapes on p...


In [94]:
# generate a TiAb dictionary
abstracts_dict = [{"input": abstract} for abstract in data.Study_content.to_list()]

# Screen literature

In [82]:
# name save directory in which the batch tasks and job results will be saved

savedir_batch = "batch-API-SR"

if not os.path.exists(f"{savedir_batch}"):
    os.mkdir(f"{savedir_batch}")
    os.mkdir(f"{savedir_batch}/results")

In [24]:
# create a directory to save pickled results

savedir_pickle = f'res-SR-openai-pickled'
if not os.path.exists(savedir_pickle):
    os.mkdir(savedir_pickle)

## Option A: batch API
per token cost half the price of on-demand calls

In [38]:
filename_workflow="fewshot_mid" # your filename suffix

### All at once (provided usage tier allows it)

In [26]:
# index = [...] # define list of indices to process if you do not intend to process the whole dataset

result_file_name_fewshot_mid, batch_id = run_batch(
    method=filename_workflow,
    savedir_batch=savedir_batch,
    abstracts_dict=abstracts_dict,
    system_prompt=system_fewshot_prompt,
    tools=tools,
    model=llm,
    job_description=f"Batch job for SR abstracts using fewshot and gpt-4.1-mini model",
    examples=examples,
    # index=index, 
    # monitor=True, # uncomment if you wish to monitor the batch job
    # recheck_time=300 # uncomment to set recheck time in seconds
)

Batch job batch_697792e43ff881909e047dca811d9259 created. No monitoring. You can check the status later using the OpenAI API. Use the file ID batch_697792e43ff881909e047dca811d9259 to retrieve the results and save them to a file name as returned by this function.


#### Fetch batch results
if necessary (when monitoring was not turned on, or an error or interruption occured)

In [29]:
# result_file_name_fewshot_mid = f"{savedir_batch}/results/batch_{filename_workflow}.jsonl"
# batch_id = "batch_697792e43ff881909e047dca811d9259"
fetch_batch_results(batch_id, result_file_name_fewshot_mid)

Batch job completed successfully.
Output file ID: file-UrpGdRgK29Aghd6wFbCPf3
Saving results to:  batch-API-SR/results/batch_fewshot_mid.jsonl


### Or process in subbatches
if usage tier limits the number of queued batch tokens

In [ ]:
filename_workflow="fewshot_mid" # your filename suffix

In [30]:
batch_size = 500 # define your batch size, between 250-1000 is reasonable

result_name_list_fewshot_mid = []
batch_id_list_fewshot_mid = []

n_batches = (len(abstracts_dict) + batch_size - 1) // batch_size

In [31]:
# Creating an array of json tasks
for i in range(n_batches):
    # if i < 3:  # uncomment and change the number to skip initial batches if they have already been processed
    #     continue
    print(f"part {i+1}")
    start_idx = i*batch_size # define index of first record to screen in current batch
    end_idx = min(len(abstracts_dict), (i+1)*batch_size) # define index of last record to screen in current batch

    print(f"Processing abstracts {start_idx} to {end_idx}...")

    index = list(range(start_idx, end_idx))

    filename_workflow_part=f"fewshot_mid-part{i+1}"

    result_file_name_fewshot_mid, batch_id = run_batch(
    method=filename_workflow_part,
    savedir_batch=savedir_batch,
    abstracts_dict=abstracts_dict,
    system_prompt=system_fewshot_prompt,
    tools=tools,
    model=llm,
    job_description=f"Batch job, part {i+1} for SR abstracts using fewshot and gpt-4.1-mini model",
    examples=examples,
    index=index,
    monitor=True, # monitoring should be enabled for subbatch processing
    recheck_time=120 # set recheck time in seconds
    )

    result_name_list_fewshot_mid.append(result_file_name_fewshot_mid)
    batch_id_list_fewshot_mid.append(batch_id)

part 1
Processing abstracts 0 to 3...
Batch job batch_69779366de748190a2faaf6c8b0e4ca2 created. Monitoring status...
job batch_69779366de748190a2faaf6c8b0e4ca2 is still running
job batch_69779366de748190a2faaf6c8b0e4ca2 is done
Batch job completed successfully.
Output file ID: file-7xhEAA1tB2KkqHWxnMQ1Tm
Saving results to:  batch-API-SR/results/batch_fewshot_mid-part1.jsonl


#### Fetch batch results
Only if necessary (when monitoring was not turned on, or an error or interruption occured)

In [54]:
part_index = '-part1' # specify part index as 'part-i' where i is the part number from the relevant subbatch, e.g., -part3 

result_file_name_fewshot_mid = f"{savedir_batch}/results/batch_{filename_workflow}{part_index}.jsonl"
# batch_id = "batch_69779366de748190a2faaf6c8b0e4ca2"

fetch_batch_results(batch_id, result_file_name_fewshot_mid)

Batch job completed successfully.
Output file ID: file-7xhEAA1tB2KkqHWxnMQ1Tm
Saving results to:  batch-API-SR/results/batch_fewshot_mid-part1.jsonl


#### Consolidate results from the subbatches into on file

In [55]:
result_name_list_fewshot_mid

['batch-API-SR/results/batch_fewshot_mid-part1.jsonl']

In [95]:
# only works if the batch function has been run completely and a list of result names is available

result_file_name_fewshot_mid = f"{savedir_batch}/results/batch_{filename_workflow}.jsonl"

with open(result_file_name_fewshot_mid, "w", encoding = "utf-8") as outfile:
    # also fix custom_id suffix
    i = 0
    for file_name in result_name_list_fewshot_mid:
        with open(file_name, "r", encoding="utf-8") as infile:
            for line in infile:
                row_data = json.loads(line)
                custom_id = row_data.get("custom_id", "")
                parts = custom_id.split("-")
                if parts[-1] != str(i):
                        parts[-1] = str(i)
                row_data["custom_id"] = "-".join(parts)

                outfile.write(json.dumps(row_data)+"\n")
                i += 1

In [ ]:
# or in bash, if custom_id is correctly indexed
# change the last part index accordingly

!bash -c "cat batch-API-SR/results/batch_fewshot-mid-SR-part{1..16}.jsonl > batch-API-SR/results/batch_fewshot-mid-SR.jsonl"

### Extract, parse and sort batch API results

In [96]:
# Usage
result_file_name_fewshot_mid = f"{savedir_batch}/results/batch_{filename_workflow}.jsonl"

sorted_fewshot_mid_index, sorted_fewshot_mid_full, sorted_fewshot_mid_extracted, sorted_fewshot_mid_content, sorted_fewshot_mid = extract_batchAPI(result_file_name_fewshot_mid)

In [97]:
# Check if all dictionaries have 10 keys
all_have_10_keys = all(len(d) == 10 for d in sorted_fewshot_mid_extracted)

# Output the result
if all_have_10_keys:
    print("All dictionaries have 10 keys.")
else:
    print("Not all dictionaries have 10 keys.")

All dictionaries have 10 keys.


### Obtain final results

In [98]:
batch_results_fewshot_mid = [None] * len(abstracts_dict)

for idx, result in fewshot_chain.batch_as_completed(sorted_fewshot_mid, config={'max_concurrency':5}):
    batch_results_fewshot_mid[idx] = result

In [99]:
batch_results_fewshot_mid

[{'evaluations': {'population_evaluation': {'decision': 'False',
    'reason': 'Study involves artificial neural organisms, not humans.'},
   'exposure_evaluation': {'decision': 'True',
    'reason': 'Exposure to varying magnetic field patterns is described.'},
   'comparison_evaluation': {'decision': 'True',
    'reason': 'Comparison between different levels of magnetic field variability is present.'},
   'outcome_evaluation': {'decision': 'True',
    'reason': 'Outcome relates to cognitive performance, relevant to the review.'},
   'study_type_evaluation': {'decision': 'False',
    'reason': 'Study type is experimental but on artificial organisms, not a human RCT.'}},
  'summary': {'True': 3, 'False': 2, 'NotEvaluable': 0},
  'PECOS': {}},
 {'evaluations': {'population_evaluation': {'decision': 'False',
    'reason': 'Study involves microbial communities, not human participants.'},
   'exposure_evaluation': {'decision': 'True',
    'reason': 'Exposure is synthetic gravity in bioreact

In [100]:
save_batch(batch_results_fewshot_mid, 
           f'batch_results_{filename_workflow}', 
           savedir_pickle)

Saving batch : batch_results_fewshot_mid


In [101]:
check_results(batch_results_fewshot_mid)

0   {'True': 3, 'False': 2, 'NotEvaluable': 0}
1   {'True': 2, 'False': 3, 'NotEvaluable': 0}
2   {'True': 2, 'False': 3, 'NotEvaluable': 0}
0 blank results. Affecting indices: [] .


## Option B: on-demand API
standard token price

Note: token usage extraction has not been implemented here. Token usage can be extracted from the batch API results (Option A).

In [64]:
llm = "gpt-4.1-mini-2025-04-14"

In [65]:
model = ChatOpenAI(
    model = llm,
    temperature = 0.0,
    #top_p=0, # We generally recommend altering this or temperature but not both.
    seed=123, # setting a seed for reproducibility
    max_tokens = 512
)

In [66]:
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{response}"),
    ]
)

few_shot_message = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# print(few_shot_message.invoke({}).to_messages())

fewshot_prompt = ChatPromptTemplate.from_messages([
    ("system", system_fewshot_prompt),
    few_shot_message,
    ("human", """
    Now, it is your turn to evaluate the following study text: {input} \n 
    You MUST call and use the ValidateForInclusion tool to format your response."""),
])

In [67]:
model = model.bind_tools([ValidateForInclusion], strict=True)

fewshot_mid_workflow = fewshot_prompt | model | parse_results_langchain | OpenAIFunctionsAgentOutputParser() | format_zeroshot_results

### Obtain final results

In [68]:
filename_workflow="fewshot_mid" # your filename suffix

In [ ]:
batch_results_fewshot_mid = [None] * len(abstracts_dict)

for idx, result in fewshot_mid_workflow.batch_as_completed(abstracts_dict, config={'max_concurrency':3}):   # adjust parallel call number (max_concurrency) if necessary
    print(idx)
    batch_results_fewshot_mid[idx] = result

    # save results every 200 records or adjust frequency. 
    # Be aware that there might be some holes due to the parallel processing. 
    # Make sure to execute the next cell, once finished, to save the complete results.

    if idx in list(range(199,len(abstracts_dict),200)):
        save_batch(batch_results_fewshot_mid, 
                f'batch_results_{filename_workflow}', 
                savedir_pickle)


0
2
1


In [70]:
save_batch(batch_results_fewshot_mid, 
           f'batch_results_{filename_workflow}', 
           savedir_pickle)

Saving batch : batch_results_fewshot_mid


In [132]:
check_results(batch_results_fewshot_mid)

0   {'True': 3, 'False': 2, 'NotEvaluable': 0}
1   {'True': 2, 'False': 3, 'NotEvaluable': 0}
2   {'True': 0, 'False': 5, 'NotEvaluable': 0}
0 blank results. Affecting indices: [] .


# Define decision rule

## Decision rule 6 - sensitive, allow manual disambiguitation
\>= 3/5 PECOS fulfilled -> include  
<3/5 PECOS fulfilled and >= 2/5 PECOS not fulfilled -> exclude  
if none of the above applies and >= 2/5 PECOS not evaluable -> tag for manual check

In [102]:
def alg6(batch_results):
    decision_restults_alg6 = []
    for i in [entry['summary'] for entry in batch_results]:
        if i['True'] >= 3:
            decision_restults_alg6.append('include')
        elif i['True'] < 3 and i['False'] >= 2:
            decision_restults_alg6.append('exclude')
        elif i['NotEvaluable'] >= 2:
            decision_restults_alg6.append('MANUAL_CHECK')
        else:
            decision_restults_alg6.append('exclude')
    return decision_restults_alg6

In [103]:
alg_functions = {
    'alg6_sens_man': alg6
}

def decision_alg(batch_results):
    results = []
    for func_name, func in alg_functions.items():
        result = {
            'algorithm': func_name,
            'predictions': func(batch_results)
        }
        results.append(result)
    return results

# Derive decisions from batch results and export labels and evaluations

## Load results

In [126]:
savedir_pickle = f'res-SR-openai-pickled' # set to your results folder
name_output = "fewshot_mid" # name of the workflow
file_path = f'{savedir_pickle}/batch_results_{name_output}.pkl'

with open(file_path, 'rb') as f:
    batch_results_fewshot_mid = pickle.load(f)


## Process results

In [127]:
# directory to save results in
savedir_res = 'res-SR-openai'

if not os.path.exists(savedir_res):
    os.mkdir(savedir_res)

In [128]:
# Generate decisions using algorithms
decisions = decision_alg(batch_results_fewshot_mid)

In [129]:
# Create expanded DataFrame from original data
expanded_df = data.copy()

name_output = "fewshot_mid" # name of the workflow
results = {} # dictionary to hold results
results[name_output] = {
    'batch_results': batch_results_fewshot_mid,
    'decisions': decisions
}

# Add evaluations columns
expanded_df[f'{name_output}_evaluations'] = [item['evaluations'] if 'evaluations' in item else None 
                                        for item in results[name_output]['batch_results']]
# Add calls columns
expanded_df[f'{name_output}_calls'] = [item['summary'] if 'summary' in item else None 
                                    for item in results[name_output]['batch_results']]

# Add decision columns from each decision rule
decisions_df = pd.DataFrame({
r['algorithm']: r['predictions'] 
for r in results[name_output]['decisions']
})

for col in decisions_df.columns:
    expanded_df[f'{name_output}_{col}'] = decisions_df[col]

# Export everything
expanded_df.to_excel(f'{savedir_res}/complete_results.xlsx', index=False)

# Add decision label back to original import

## Option A: Export back to excel

In [122]:
df['Label'] = decisions[0]['predictions']

In [123]:
df

,Title,Abstract,Title_expl,Abstract_expl,Label
0,Magnetic field variability and cognitive perfo...,OBJECTIVE: To explore whether ambient magnetic...,Title: Magnetic field variability and cognitiv...,Abstract: OBJECTIVE: To explore whether ambien...,include
1,Microbial adaptation to synthetic gravity in o...,OBJECTIVE: To assess whether microbial communi...,Title: Microbial adaptation to synthetic gravi...,Abstract: OBJECTIVE: To assess whether microbi...,exclude
2,Effects of algorithmic soundscapes on plant gr...,OBJECTIVE: To determine whether exposure to al...,Title: Effects of algorithmic soundscapes on p...,Abstract: OBJECTIVE: To determine whether expo...,exclude


In [ ]:
df.drop(columns=['Title_expl', 'Abstract_expl'])
df

In [124]:
df.to_excel(f'{savedir_res}/abs-screen_labeled.xlsx', index=False)

## Option B: Export back to ris

In [110]:
df['LB'] = decisions[0]['predictions']

In [111]:
df.drop(columns=['Title_expl', 'Abstract_expl'])
df

,AB,AN,DA,DP,ET,IS,J2,KW,LA,LB,...,SN,SP,ST,T2,TI,VL,ID,Title_expl,Abstract_expl,Label
0,OBJECTIVE: To explore whether ambient magnetic...,1000001,Jan,Nlm,2031/01/10,1,Journal of speculative computational science,Magnetic Fields Neural Networks Simulation Cog...,eng,include,...,0000-0001 (Print) 0000-0001,1-5,Magnetic field variability and cognitive perfo...,J Spec Comput Sci,Magnetic field variability and cognitive perfo...,12,50001,Title: Magnetic field variability and cognitiv...,Abstract: OBJECTIVE: To explore whether ambien...,include
1,OBJECTIVE: To assess whether microbial communi...,1000002,Apr,Nlm,2042/04/03,2,International journal of astro-biology experim...,Microbiology Gravity Simulation Evolution Bior...,eng,exclude,...,0000-0002 (Print) 0000-0002,77-82,Microbial adaptation to synthetic gravity in o...,Int J Astro-Biol Exp,Microbial adaptation to synthetic gravity in o...,5,50002,Title: Microbial adaptation to synthetic gravi...,Abstract: OBJECTIVE: To assess whether microbi...,exclude
2,OBJECTIVE: To determine whether exposure to al...,1000003,Oct,Nlm,2037/10/21,4,Journal of experimental bioacoustics,Bioacoustics Plant Growth Algorithms Sound Exp...,eng,exclude,...,0000-0003 (Print) 0000-0003,210-214,Effects of algorithmic soundscapes on plant gr...,J Exp Bioacoust,Effects of algorithmic soundscapes on plant gr...,9,50003,Title: Effects of algorithmic soundscapes on p...,Abstract: OBJECTIVE: To determine whether expo...,exclude


In [112]:
write_ris(df, f'{savedir_res}/abs-screen_labeled.ris')